# 03. Phase B: Hybrid Longitudinal SimCLR + Auxiliary $\Delta t$ Adaptation & Phase A vs. B Benchmark

In **Phase B**, we adapt the pretrained 3D MedicalNet ResNet-18 backbone to longitudinal MS brain MRI across **18 patients** (all 14 unlabeled test patients + 4 active training fold patients, keeping the held-out validation patient 100% isolated) before training the segmentation decoder:

1. **Partial Unfreezing:** Freeze `layer1–layer3` (preserving fine edge detectors) and unfreeze `conv1` + `bn1` (4-modality stem) and `layer4` (high-level semantics).
2. **Patient-Masked NT-Xent Contrastive Loss:** Pulls anatomically aligned $64^3$ foreground brain patches $(p_{t_1}, p_{t_2})$ from the same patient across visits together while masking out same-patient false negatives in the batch.
3. **Auxiliary $\Delta t$ Difference Head:** Predicts $\widehat{\Delta t}$ from $[z_{t_2} - z_{t_1}, |z_{t_2} - z_{t_1}|]$ so longitudinal progression signals never collapse to zero.
4. **Apples-to-Apples Downstream Comparison:** Freezes the Phase B adapted backbone, trains a fresh `SiameseLongitudinalUNet3D` decoder under the exact same protocol as Phase A, and generates the final portfolio benchmark.

In [ ]:
# 1. Mount Google Drive & Setup Environment (Colab Pro A100)
import json
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'monai>=1.3.0', 'nibabel>=5.0.0', 'pyyaml', 'tqdm'])

REPO_CANDIDATES = [
    Path('/content/drive/MyDrive/masters/simclr-ms-mri'),
    Path('/content/simclr-ms-mri'),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
]
for candidate in REPO_CANDIDATES:
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate.resolve()
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break

import torch
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

In [ ]:
# 2. Build Zero-Leakage LOPO Split & Preload All 19 Patients into Colab Pro System RAM
from src.utils.config import load_config, set_deterministic_seed
from src.data.isbi_dataset import discover_isbi2015_visits, get_lopo_fold_splits
from src.data.transforms import (
    InMemoryVolumeCache,
    AlignedLongitudinalSimCLRDataset,
    LongitudinalSegmentationPatchDataset,
)

cfg = load_config(PROJECT_ROOT / 'configs' / 'default.yaml')
set_deterministic_seed(cfg['project']['seed'])

FOLD_IDX = 0
patient_visits = discover_isbi2015_visits(cfg['data']['root_dir'], modalities=cfg['data']['modalities'])
split = get_lopo_fold_splits(patient_visits, fold_idx=FOLD_IDX, max_time_gap_norm=cfg['data']['max_time_gap_norm'])
split.verify_zero_leakage()

print(f'Phase B SimCLR Pool: {len(split.simclr_patient_ids)} patients ({len(split.simclr_pairs)} visit pairs)')
print(f'Held-Out Val Subject (Excluded from SimCLR & Seg Train): {split.val_patient_id}')

volume_cache = InMemoryVolumeCache(
    modalities=cfg['data']['modalities'],
    target_rater=cfg['data']['target_rater'],
    enabled=(cfg['data']['cache_rate'] > 0),
)
all_visits = [v for visits in patient_visits.values() for v in visits]
volume_cache.preload_visits(all_visits)
print(f'Cached all {len(all_visits)} 3D scans in system RAM (~9.2 GB).')

simclr_ds = AlignedLongitudinalSimCLRDataset(
    pairs=split.simclr_pairs,
    volume_cache=volume_cache,
    patch_size=tuple(cfg['data']['patch_size']),
    patches_per_pair=cfg['data']['patches_per_pair_simclr'],
    max_spatial_jitter=cfg['data']['max_spatial_jitter'],
    augment=True,
)
simclr_loader = DataLoader(
    simclr_ds,
    batch_size=cfg['simclr']['batch_size'],
    shuffle=True,
    num_workers=cfg['data']['num_workers'],
    pin_memory=True,
    drop_last=True,
)

In [ ]:
# 3. Phase B Step 1: Run Hybrid Longitudinal SimCLR + Auxiliary Δt Pretraining
import matplotlib.pyplot as plt
from src.models.backbone import MedicalNet3DResNet18
from src.models.simclr import LongitudinalSimCLR3D
from src.engine.trainer import fit_longitudinal_simclr

OUTPUT_DIR = Path(cfg['project']['output_dir'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
simclr_backbone_path = OUTPUT_DIR / f'phase_b_simclr_backbone_fold{FOLD_IDX}.pt'

backbone_phase_b = MedicalNet3DResNet18(
    in_channels=cfg['backbone']['in_channels'],
    pretrained=cfg['backbone']['pretrained'],
)
simclr_model = LongitudinalSimCLR3D(
    backbone=backbone_phase_b,
    proj_hidden_dim=cfg['simclr']['proj_hidden_dim'],
    proj_out_dim=cfg['simclr']['proj_out_dim'],
    unfreeze_layers=cfg['backbone']['unfreeze_layers_phase_b'],
)

stats = simclr_model.backbone.count_trainable_parameters()
print(f"Phase B Unfrozen Backbone Params (conv1 + bn1 + layer4): {stats['trainable']:,} / {stats['total']:,}")

simclr_history = fit_longitudinal_simclr(
    model=simclr_model,
    dataloader=simclr_loader,
    epochs=cfg['simclr']['epochs'],
    lr=cfg['simclr']['lr'],
    weight_decay=cfg['simclr']['weight_decay'],
    temperature=cfg['simclr']['temperature'],
    lambda_delta_t=cfg['simclr']['lambda_delta_t'],
    device=DEVICE,
    use_amp=cfg['simclr']['use_amp'],
    amp_dtype_str=cfg['simclr']['amp_dtype'],
    save_path=simclr_backbone_path,
)

# Plot SimCLR Convergence & Top-1 Longitudinal Patch Retrieval Accuracy
epochs_x = [h['epoch'] for h in simclr_history]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(epochs_x, [h['nt_xent_loss'] for h in simclr_history], color='#5C7CFA', lw=2)
axes[0].set_title('Patient-Masked NT-Xent Loss (↓)')
axes[1].plot(epochs_x, [h['delta_t_loss'] for h in simclr_history], color='#FF922B', lw=2)
axes[1].set_title('Auxiliary Δt Difference Loss (↓)')
axes[2].plot(epochs_x, [h['top1_acc'] * 100 for h in simclr_history], color='#20C997', lw=2)
axes[2].set_title('Longitudinal Patch Retrieval Top-1 Acc % (↑)')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Phase B Step 2: Freeze SimCLR-Adapted Backbone & Train Downstream Segmentation Decoder
from src.models.segmentation_decoder import SiameseLongitudinalUNet3D
from src.losses.segmentation_loss import EvolutionWeightedDiceFocalLoss
from src.engine.trainer import train_segmentation_epoch
from src.engine.evaluator import evaluate_lopo_validation_pairs

set_deterministic_seed(cfg['project']['seed'])
train_seg_ds = LongitudinalSegmentationPatchDataset(
    pairs=split.seg_train_pairs,
    volume_cache=volume_cache,
    patch_size=tuple(cfg['data']['patch_size']),
    patches_per_pair=cfg['data']['patches_per_pair_seg'],
    pos_neg_ratio=cfg['data']['pos_neg_ratio'],
    augment=True,
)
train_seg_loader = DataLoader(
    train_seg_ds,
    batch_size=cfg['segmentation']['batch_size'],
    shuffle=True,
    num_workers=cfg['data']['num_workers'],
    pin_memory=True,
)

backbone_phase_b.load_state_dict(torch.load(simclr_backbone_path, map_location=DEVICE))
backbone_phase_b.freeze_all()

model_phase_b = SiameseLongitudinalUNet3D(
    backbone=backbone_phase_b,
    decoder_channels=tuple(cfg['segmentation']['decoder_channels']),
    freeze_backbone=True,
).to(DEVICE)

criterion = EvolutionWeightedDiceFocalLoss(
    dice_weight=cfg['segmentation']['dice_weight'],
    focal_weight=cfg['segmentation']['focal_weight'],
    change_voxel_weight=cfg['segmentation']['change_voxel_weight'],
)
optimizer = torch.optim.AdamW(
    [p for p in model_phase_b.parameters() if p.requires_grad],
    lr=cfg['segmentation']['lr_decoder'],
    weight_decay=cfg['segmentation']['weight_decay'],
)
epochs = cfg['segmentation']['epochs']
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_ckpt_b = OUTPUT_DIR / f'phase_b_fold{FOLD_IDX}_best.pt'
best_val_dice_b = -1.0
best_val_summary_b = {}

for epoch in range(1, epochs + 1):
    train_metrics = train_segmentation_epoch(
        model=model_phase_b,
        dataloader=train_seg_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        use_amp=cfg['segmentation']['use_amp'],
        amp_dtype_str=cfg['segmentation']['amp_dtype'],
    )
    scheduler.step()

    if epoch % 5 == 0 or epoch == epochs:
        val_summary, _ = evaluate_lopo_validation_pairs(
            model=model_phase_b,
            val_pairs=split.seg_val_pairs,
            volume_cache=volume_cache,
            device=DEVICE,
            roi_size=tuple(cfg['data']['patch_size']),
            sw_batch_size=cfg['segmentation']['sw_batch_size'],
            overlap=cfg['segmentation']['sw_overlap'],
            threshold=cfg['segmentation']['threshold'],
            use_amp=cfg['segmentation']['use_amp'],
            amp_dtype_str=cfg['segmentation']['amp_dtype'],
        )
        print(
            f"[Epoch {epoch:02d}/{epochs:02d}] Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Full-3D t2 Dice: {val_summary['t2_dice']:.4f} | "
            f"New/Enlarging Dice: {val_summary['new_enlarging_dice']:.4f} | "
            f"ΔVol MAE: {val_summary['delta_vol_mae_ml']:.3f} mL"
        )
        if val_summary['t2_dice'] > best_val_dice_b:
            best_val_dice_b = val_summary['t2_dice']
            best_val_summary_b = val_summary
            torch.save(model_phase_b.state_dict(), best_ckpt_b)

with (OUTPUT_DIR / f'phase_b_fold{FOLD_IDX}_metrics.json').open('w') as f:
    json.dump(best_val_summary_b, f, indent=2)

In [ ]:
# 5. Portfolio Benchmark: Phase A (Frozen) vs. Phase B (Longitudinal SimCLR + Δt)
from src.engine.evaluator import predict_full_volume_pair
from src.utils.visualization import plot_phase_comparison_bars, plot_lesion_evolution_comparison

phase_a_metrics_file = OUTPUT_DIR / f'phase_a_fold{FOLD_IDX}_metrics.json'
if phase_a_metrics_file.exists():
    with phase_a_metrics_file.open('r') as f:
        phase_a_summary = json.load(f)
else:
    phase_a_summary = {'t2_dice': 0.0, 'new_enlarging_dice': 0.0, 'stable_dice': 0.0, 'delta_vol_mae_ml': 0.0}

print('=== HELD-OUT PATIENT LONGITUDINAL BENCHMARK (PHASE A vs PHASE B) ===')
for key in ['t2_dice', 'new_enlarging_dice', 'stable_dice', 'delta_vol_mae_ml']:
    va = phase_a_summary.get(key, 0.0)
    vb = best_val_summary_b.get(key, 0.0)
    print(f'{key:<22} | Phase A: {va:.4f} | Phase B (SimCLR): {vb:.4f} | Δ: {vb - va:+.4f}')

fig_bars = plot_phase_comparison_bars(phase_a_summary, best_val_summary_b)
plt.show()

# Side-by-side 5-panel visual comparison on held-out patient visit pair
val_pair = split.seg_val_pairs[0]
d1 = volume_cache.get_visit_data(val_pair.visit_t1)
d2 = volume_cache.get_visit_data(val_pair.visit_t2)

model_phase_b.load_state_dict(torch.load(best_ckpt_b, map_location=DEVICE))
pred_b = predict_full_volume_pair(
    model=model_phase_b,
    vol_t1=d1['image'],
    vol_t2=d2['image'],
    delta_t_norm=val_pair.delta_t_norm,
    device=DEVICE,
    roi_size=tuple(cfg['data']['patch_size']),
)

phase_a_ckpt = OUTPUT_DIR / f'phase_a_fold{FOLD_IDX}_best.pt'
if phase_a_ckpt.exists():
    backbone_a = MedicalNet3DResNet18(in_channels=cfg['backbone']['in_channels'], pretrained=False)
    model_phase_a = SiameseLongitudinalUNet3D(backbone=backbone_a, decoder_channels=tuple(cfg['segmentation']['decoder_channels'])).to(DEVICE)
    model_phase_a.load_state_dict(torch.load(phase_a_ckpt, map_location=DEVICE))
    pred_a_mask = predict_full_volume_pair(model_phase_a, d1['image'], d2['image'], val_pair.delta_t_norm, DEVICE, tuple(cfg['data']['patch_size']))['pred_mask_t2']
else:
    pred_a_mask = pred_b['pred_mask_t2']

fig_evol = plot_lesion_evolution_comparison(
    flair_t1=d1['image'][0],
    flair_t2=d2['image'][0],
    gt_mask_t1=d1['mask'],
    gt_mask_t2=d2['mask'],
    pred_phase_a_t2=pred_a_mask,
    pred_phase_b_t2=pred_b['pred_mask_t2'],
    patient_pair_title=f'Phase A vs. Phase B Longitudinal SimCLR ({val_pair.pair_id})',
)
plt.show()